# Task 2.1: Evaluation metrics

Accuracy metrics from the lecture slides, implemented from scratch in NumPy so that every model from
Task 1 (individual RecBole models and our hybrids) is evaluated by the same code.

The code lives in two modules so every task imports the same implementation:
[metrics.py](metrics.py) (the metrics, pure NumPy) and [recbole_utils.py](recbole_utils.py) (Task 1 split, model building, score matrices).

**Layout**

1. Rating-prediction metrics: MAE, MSE, RMSE
2. Top-K metrics, per user: Precision, Recall, F1, Hit, NDCG, MRR, AP (for MAP), Fallout
3. AUC and the averaged ROC curve
4. `evaluate_scores(...)` / `evaluate_topk(...)`: compute everything over all users at once
5. Getting score matrices out of RecBole
6. Sanity checks: hand-worked examples, scikit-learn, RecBole's metric code, RecBole's own evaluation of EASE

Beyond-accuracy metrics (diversity, novelty, calibration, coverage, exposure, fairness, popularity bias)
are in [task_2_1_beyond_accuracy.ipynb](task_2_1_beyond_accuracy.ipynb).

The models themselves are evaluated with this code in [task_2_2.ipynb](task_2_2.ipynb).

Run with the `RecBole` environment from `requirements.txt` (Python 3.10, NumPy 1.23.5).

In [ ]:
import numpy as np
import pandas as pd
from recbole.utils import get_model, get_trainer

from metrics import *  # noqa: F403  (all metric functions, K, KS, trapezoid)
from recbole_utils import load_split, model_scores

## 1. Rating-prediction metrics

Implemented in [metrics.py](metrics.py): `mae`, `mse`, `rmse`.

## 2. Top-K metrics (per user)

All top-K metrics are computed from one **relevance vector**: `rel[i] = 1` if the item at rank
`i+1` of the user's ranking is in $G$. `n_rel` is $|G|$. The public functions take a ranked list of
item ids and a set of relevant ids, so they can be used on any model's output.

Implemented in [metrics.py](metrics.py): `precision_at_k`, `recall_at_k`, `hit_at_k`, `f1_at_k`, `fallout_at_k`, `ndcg_at_k`, `mrr_at_k`, `ap_at_k` (public, take a ranked list of ids and a set of relevant ids), built on per-user helpers that take the relevance vector (`_precision`, `_ndcg`, ...).

## 3. AUC and ROC

**AUC** for one user is the probability that a randomly chosen held-out positive is scored above a
randomly chosen unseen negative, with ties counting ½. It is computed with the Mann-Whitney U statistic
on average ranks, which is exact and $O(n\log n)$.

**ROC.** For one user, moving $K$ from 0 to the number of candidates traces
(fallout@K, recall@K) from (0, 0) to (1, 1). To average over users, each user's step curve is evaluated
on a common fallout grid and the recall values are averaged (vertical averaging). The area under the
averaged curve equals the mean per-user AUC, up to the grid resolution and tie handling.

Implemented in [metrics.py](metrics.py): `auc_user`, `roc_points`, `recall_on_grid`.

## 4. Evaluating a whole model

Two entry points:

* **`evaluate_scores(scores, test_items, ks)`**: the full score matrix `scores[user, item]`, where
  non-candidate items (training history, padding) have score `-inf`. This computes **everything**,
  including fallout, AUC and the ROC curve, because it knows every candidate's score.
* **`evaluate_topk(recs, test_items, ks)`**: only top-K lists (`{user: [item, ...]}`), for example
  from a CSV a model wrote out. Fallout, AUC and ROC need the full candidate set, so they are not
  computed here.

`test_items` maps a user to the collection of their held-out items, as row/column indices of
`scores` (for `evaluate_scores`) or as whatever ids `recs` uses (for `evaluate_topk`).

Implemented in [metrics.py](metrics.py): `evaluate_scores`, `evaluate_topk`, and `TOPK_METRICS` (name → per-user function).

## 5. Score matrices from RecBole

[recbole_utils.py](recbole_utils.py) builds the Task 1 split (`load_split`), builds and trains models
(`build_model`), and turns any score function into a `[n_users, n_items]` matrix (`score_matrix`,
`model_scores`). That function can be `model.full_sort_predict` or a hybrid. The padding item and each
user's history (train for validation; train + valid for test) are set to `-inf`, which is the candidate
convention above. Section 7.5 checks this glue code end to end.

## 6. Sanity checks

### 6.1 Hand-worked example

Ranking `[a, b, c, d, e]`, relevant items $G=\{a, c, f\}$, $K=5$. Hits are at ranks 1 and 3.

* P@5 = 2/5, R@5 = 2/3, F1 = 2·0.4·0.667/1.067 = 0.5, Hit@5 = 1
* DCG = 1/log₂2 + 1/log₂4 = 1.5. IDCG over min(5, 3) = 3 items = 1 + 0.631 + 0.5 = 2.131. NDCG = 0.704
* MRR = 1
* AP: P@1·1 + P@3·1 = 1 + 2/3. Divided by min(5, 3) = 3 this is 0.556.

In [ ]:
ranked, relevant = list("abcde"), {"a", "c", "f"}
assert np.isclose(precision_at_k(ranked, relevant, 5), 2 / 5)
assert np.isclose(recall_at_k(ranked, relevant, 5), 2 / 3)
assert np.isclose(f1_at_k(ranked, relevant, 5), 0.5)
assert hit_at_k(ranked, relevant, 5) == 1 and hit_at_k(ranked, {"z"}, 5) == 0
assert np.isclose(ndcg_at_k(ranked, relevant, 5), 1.5 / (1 + 1 / np.log2(3) + 0.5))
assert np.isclose(mrr_at_k(ranked, relevant, 5), 1.0)
assert np.isclose(ap_at_k(ranked, relevant, 5), (1 + 2 / 3) / 3)
assert mrr_at_k(ranked, {"z"}, 5) == 0 and ap_at_k(ranked, {"z"}, 5) == 0  # no hit -> 0
assert np.isclose(mae([4, 3, 5], [3.5, 3, 4]), 0.5)
assert np.isclose(rmse([4, 3, 5], [3.5, 3, 4]), np.sqrt((0.25 + 0 + 1) / 3))
assert np.isclose(ndcg_at_k(ranked, relevant, 5, gains={i: 1 for i in relevant}), ndcg_at_k(ranked, relevant, 5))
print("hand-worked examples OK")

hand-worked examples OK


### 6.2 The MAP example from the slides: three denominators

$|G| = 5$, $K = 10$, hits at ranks 1, 3 and 6, so the precisions at the hits are 1, 2/3 and 1/2
(sum 2.167).

* slide (÷ hits = 3): 0.722
* ÷ $|G|$ = 5: 0.433
* **ours, ÷ min(K, |G|) = 5: 0.433**

With $K = 3$ and hits at ranks 1 and 3, `min` divides by 3 while `relevant` still divides by 5.
That is the case where $|G|$ would unfairly cap the score.

In [ ]:
ranked = ["h1", "x", "h2", "x2", "x3", "h3", "x4", "x5", "x6", "x7"]
relevant = {"h1", "h2", "h3", "m1", "m2"}
for denom in ["hits", "relevant", "min"]:
    print(f"{denom:>8}: K=10 -> {ap_at_k(ranked, relevant, 10, denom):.3f}   K=3 -> {ap_at_k(ranked, relevant, 3, denom):.3f}")

    hits: K=10 -> 0.722   K=3 -> 0.833
relevant: K=10 -> 0.433   K=3 -> 0.333
     min: K=10 -> 0.433   K=3 -> 0.556


### 6.3 AUC and fallout against scikit-learn

The AUC check includes ties.

In [4]:
from sklearn.metrics import roc_auc_score, roc_curve

rng = np.random.default_rng(0)
for _ in range(200):
    n = rng.integers(5, 60)
    y = rng.random(n) < 0.3
    if y.all() or not y.any():
        continue
    s = rng.integers(0, 6, n).astype(float)  # many ties on purpose
    assert np.isclose(auc_user(s[y], s[~y]), roc_auc_score(y, s))

# hand example: 0.9 beats all 3 negatives, 0.4 beats 0.1 and ties 0.4 -> (3 + 1 + 0.5) / 6
assert np.isclose(auc_user([0.9, 0.4], [0.8, 0.1, 0.4]), 0.75)

# fallout@K from a full ranking == FP / (FP + TN)
ranked = list("abcdefgh"); relevant = {"b", "e"}
assert np.isclose(fallout_at_k(ranked, relevant, 3), 2 / 6)

# area under one user's ROC curve == that user's AUC (no ties)
s = rng.random(50); y = rng.random(50) < 0.2
order = np.argsort(-s)
f, r = roc_points(y[order])
assert np.isclose(trapezoid(r, f), roc_auc_score(y, s))
print("AUC / ROC checks OK")

AUC / ROC checks OK


### 6.4 Against RecBole's metric code

RecBole's metric classes compute from a boolean `pos_index[user, rank]` matrix and `pos_len = |G|`.
On random data they must give the same numbers as ours. (Older RecBole code uses `np.float` /
`np.bool`, which NumPy 2 removed; the aliases are restored temporarily if missing, so this cell also
runs outside the project environment.)

In [ ]:
from recbole.evaluator import metrics as rb

_patched = [a for a in ("float", "bool") if a not in vars(np)]
for a in _patched:
    setattr(np, a, {"float": float, "bool": bool}[a])
try:
    rng = np.random.default_rng(1)
    max_k, n_users = 20, 300
    pos_len = rng.integers(1, 40, n_users)
    pos_index = np.zeros((n_users, max_k), dtype=bool)
    for u in range(n_users):  # a user can't have more hits than relevant items
        hits = rng.choice(max_k, size=min(pos_len[u], rng.integers(0, max_k)), replace=False)
        pos_index[u, hits] = True

    theirs = {
        "precision": rb.Precision.metric_info(None, pos_index),
        "recall": rb.Recall.metric_info(None, pos_index, pos_len),
        "ndcg": rb.NDCG.metric_info(None, pos_index, pos_len),
        "mrr": rb.MRR.metric_info(None, pos_index),
        "hit": rb.Hit.metric_info(None, pos_index),
        "map": rb.MAP.metric_info(None, pos_index, pos_len),
    }
    for name, their_vals in theirs.items():
        for k in (1, 5, 10, 20):
            ours = np.mean([TOPK_METRICS[name](pos_index[u].astype(float), pos_len[u], k) for u in range(n_users)])
            assert np.isclose(ours, their_vals[:, k - 1].mean()), (name, k, ours, their_vals[:, k - 1].mean())
    print("matches RecBole for", ", ".join(theirs))
finally:
    for a in _patched:
        delattr(np, a)

matches RecBole for precision, recall, ndcg, mrr, hit, map


### 6.5 End to end against RecBole's evaluation of a trained model

EASE is fitted when the model is built (no training loop, no randomness), so it is a cheap end-to-end
check. On the Task 1 test split, `evaluate_scores` and RecBole's `trainer.evaluate` must agree on
Recall, MRR, NDCG, Hit and Precision@10. This checks the split loading, the history masking and the
id mapping in Section 6, not just the metric formulas.

In [6]:
config, dataset, train_data, valid_data, test_data = load_split("EASE")
ease = get_model("EASE")(config, train_data.dataset).to(config["device"])
theirs = get_trainer(config["MODEL_TYPE"], "EASE")(config, ease).evaluate(test_data, load_best_model=False, show_progress=False)

scores, test_items = model_scores(ease, dataset, test_data)
ease_result = evaluate_scores(scores, test_items, ks=(K,))
ours = ease_result["topk"][f"@{K}"]

check = pd.DataFrame({
    "ours": [ours[m] for m in ["recall", "mrr", "ndcg", "hit", "precision"]],
    "RecBole": [theirs[f"{m}@{K}"] for m in ["recall", "mrr", "ndcg", "hit", "precision"]],
}, index=[f"{m}@{K}" for m in ["recall", "mrr", "ndcg", "hit", "precision"]])
display(check.round(4))
assert np.allclose(check["ours"], check["RecBole"], atol=1e-4), "differs from RecBole's evaluation"
print(f"EASE matches RecBole on the Task 1 test split ({len(test_items)} users)")

,ours,RecBole
recall@10,0.2805,0.2805
mrr@10,0.5277,0.5277
ndcg@10,0.3295,0.3295
hit@10,0.8197,0.8197
precision@10,0.2206,0.2206


EASE matches RecBole on the Task 1 test split (943 users)
